# Lab 1F-1: Classical MDM / Classical ML Retrieval Baseline

Exercise 1.F compares two retrieval paradigms on Flickr8k. This notebook implements Pipeline 1: a classical MDM / classical ML pipeline with structured metadata, caption TF-IDF, handcrafted HSV color histograms, edge-density descriptors, k-NN/exact nearest-neighbor visual retrieval, exact cosine similarity, and late score fusion only for true dual text+image queries.

Pipeline 1.F-1 is the classical non-deep-learning pipeline. It uses explicit features and includes k-NN over handcrafted visual descriptors as an interpretable classical ML retrieval component. It excludes deep-learning representation learning, not machine learning.

The architecture follows the fixed Part E baseline: offline ingestion path, online query path, media/content DB, structured store, semi-structured store, feature/descriptor DB, query processor/query engine, and results/visualization.

## 0. Setup

Set `FAST_DEV=True` for a 200-image subset used during debugging. Set it to `False` only when running the full Flickr8k evaluation for final outputs.

In [ ]:
FAST_DEV = True
FAST_DEV_LIMIT = 200
K_VALUES = [1, 5, 10, 20, 50]

from pathlib import Path
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import ImageFilter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

try:
    import cv2
except ImportError:
    cv2 = None

LAB_DIR = Path('content/lab/1F')
if not (LAB_DIR / 'evaluation_queries.json').exists():
    LAB_DIR = Path('.')
OUTPUT_DIR = LAB_DIR / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
QUERY_FILE = LAB_DIR / 'evaluation_queries.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'FAST_DEV={FAST_DEV}')

## 1. Offline Ingestion Path

Dataset loading is the multimedia source step. The resulting dataframe separates structured columns, semi-structured caption lists, and media/content objects.

In [ ]:
from datasets import load_dataset, concatenate_datasets

print('Loading Flickr8k...')
ds = load_dataset('jxie/flickr8k')
full_ds = concatenate_datasets([ds['train'], ds['test']])
if FAST_DEV:
    full_ds = full_ds.select(range(min(FAST_DEV_LIMIT, len(full_ds))))
print(f'Images available in this run: {len(full_ds)}')

In [ ]:
def normalize_text(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()


def caption_list(item):
    caps = item.get('caption', [])
    if isinstance(caps, str):
        caps = [caps]
    elif not caps:
        caps = [item.get(f'caption_{i}', '') for i in range(5)]
    cleaned = []
    for c in caps:
        if isinstance(c, dict):
            c = c.get('raw') or c.get('text') or ''
        s = str(c).strip()
        if s:
            cleaned.append(s)
    return cleaned


def stable_image_id(item, idx, image):
    for key in ('image_id', 'img_id', 'filename', 'file_name'):
        value = item.get(key)
        if value:
            return Path(str(value)).name
    filename = getattr(image, 'filename', '')
    if filename:
        return Path(filename).name
    captions = caption_list(item)
    first_caption = captions[0] if captions else ''
    w, h = image.size
    digest = hashlib.sha1(f'{first_caption}|{w}x{h}'.encode('utf-8')).hexdigest()[:12]
    return f'flickr8k_{digest}'


records = []
for idx in range(len(full_ds)):
    item = full_ds[idx]
    image = item['image'].convert('RGB')
    w, h = image.size
    captions = caption_list(item)
    records.append({
        'image_id': stable_image_id(item, idx, image),
        'dataset_pos': idx,
        'width': w,
        'height': h,
        'aspect_ratio': round(w / h, 4),
        'caption_count': len(captions),
        'captions': captions,
        'all_captions': ' '.join(captions),
    })

df = pd.DataFrame(records)
dup_count = int(df['image_id'].duplicated().sum())
if dup_count:
    warnings.warn(f'Found {dup_count} duplicate image rows; keeping first occurrence per image_id.')
    df = df.drop_duplicates(subset='image_id', keep='first').reset_index(drop=True)
image_id_to_pos = dict(zip(df['image_id'], df['dataset_pos']))

signature = {
    'fast_dev': FAST_DEV,
    'count': int(len(df)),
    'first_items': df[['image_id', 'width', 'height', 'captions']].head(5).to_dict(orient='records'),
}
signature_path = OUTPUT_DIR / ('dataset_signature_fast_dev.json' if FAST_DEV else 'dataset_signature_full.json')
if signature_path.exists():
    old_signature = json.loads(signature_path.read_text(encoding='utf-8'))
    if old_signature.get('first_items') != signature['first_items']:
        warnings.warn(f'Dataset signature changed: {signature_path}')
signature_path.write_text(json.dumps(signature, indent=2), encoding='utf-8')
df.head()

## 2. Part E Architecture Mapping

Use this exact code-to-architecture mapping for Pipeline 1:

| Code artifact | Fixed Part E architecture component |
|---|---|
| dataset loading | multimedia sources |
| dataframe columns such as width, height, aspect ratio, caption count | structured store |
| Flickr8k caption lists | semi-structured store |
| image files/PIL images | media/content DB |
| TF-IDF fitting over captions | offline text descriptor extraction |
| HSV histogram and edge-density extraction | offline visual descriptor extraction |
| Canny edge detection | optional nested visual low-level sub-step, not a top-level architecture module |
| `tfidf_matrix` and `visual_matrix` | feature/descriptor DB |
| transforming a text query | online reuse of the text descriptor extractor |
| extracting features from query image | online reuse of the visual descriptor extractor |
| k-NN index/search over handcrafted descriptors | query processor/query engine; classical instance-based retrieval |
| cosine similarity, ranking, and late score fusion | query processor/query engine |
| result grids and P@K/R@K plots | results/visualization |

## 3. Offline Text Descriptor Extraction

Caption lists form the semi-structured store. The fitted vectorizer and sparse matrix form the text side of the feature/descriptor DB.

In [ ]:
def rank_scores(scores, exclude_image_id=None, top_k=10):
    ranked = np.argsort(scores)[::-1]
    results = []
    for idx in ranked:
        image_id = df.iloc[int(idx)]['image_id']
        if exclude_image_id is not None and image_id == exclude_image_id:
            continue
        results.append({'image_id': image_id, 'score': float(scores[int(idx)]), 'rank': len(results) + 1})
        if len(results) >= top_k:
            break
    return results


tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(df['all_captions'])
print(f'tfidf_matrix shape: {tfidf_matrix.shape}')


def rank_text_only(query_text, exclude_image_id=None, top_k=10):
    query_vec = tfidf.transform([query_text])
    scores = cosine_similarity(query_vec, tfidf_matrix).ravel()
    return rank_scores(scores, exclude_image_id=exclude_image_id, top_k=top_k)

## 4. Offline Visual Descriptor Extraction

The visual descriptor is deliberately handcrafted: HSV histogram plus edge density. Edge detection is a nested low-level visual sub-step, not a separate top-level module.

In [ ]:
def extract_hsv_histogram(image, bins=(8, 8, 8)):
    image = image.convert('RGB').resize((224, 224))
    if cv2 is not None:
        arr = np.asarray(image)
        hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
        hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256]).astype('float32').ravel()
    else:
        hsv = image.convert('HSV')
        arr = np.asarray(hsv)
        hist, _ = np.histogramdd(arr.reshape(-1, 3), bins=bins, range=((0, 255), (0, 255), (0, 255)))
        hist = hist.astype('float32').ravel()
    norm = np.linalg.norm(hist)
    return hist / norm if norm else hist


def extract_edge_density(image):
    gray = image.convert('L').resize((224, 224))
    if cv2 is not None:
        edges = cv2.Canny(np.asarray(gray), 100, 200)
        density = float(np.mean(edges > 0))
    else:
        edges = gray.filter(ImageFilter.FIND_EDGES)
        density = float(np.mean(np.asarray(edges) > 32))
    return np.array([density], dtype='float32')


def extract_visual_descriptor(image):
    return np.concatenate([extract_hsv_histogram(image), extract_edge_density(image)]).astype('float32')


visual_matrix = np.vstack([extract_visual_descriptor(full_ds[int(pos)]['image']) for pos in df['dataset_pos']])
visual_matrix = visual_matrix / np.maximum(np.linalg.norm(visual_matrix, axis=1, keepdims=True), 1e-12)
print(f'visual_matrix shape: {visual_matrix.shape}')

# Classical instance-based retrieval over explicit handcrafted descriptors.
# This is k-NN over HSV histogram + edge-density features, not representation learning.
visual_knn = NearestNeighbors(n_neighbors=min(len(df), max(K_VALUES) + 1), metric='cosine', algorithm='brute')
visual_knn.fit(visual_matrix)


def visual_knn_scores(query_image):
    query_vec = extract_visual_descriptor(query_image).reshape(1, -1)
    query_vec = query_vec / np.maximum(np.linalg.norm(query_vec, axis=1, keepdims=True), 1e-12)
    distances, indices = visual_knn.kneighbors(query_vec, n_neighbors=len(df), return_distance=True)
    scores = np.full(len(df), -np.inf, dtype='float32')
    scores[indices[0]] = 1.0 - distances[0]
    return scores


def rank_visual_knn(query_image, exclude_image_id=None, top_k=10):
    scores = visual_knn_scores(query_image)
    return rank_scores(scores, exclude_image_id=exclude_image_id, top_k=top_k)


def rank_fused(query_text, query_image, text_weight=0.5, exclude_image_id=None, top_k=10):
    if query_text is None or query_image is None:
        raise ValueError('Fused retrieval requires both query_text and query_image.')
    text_scores = cosine_similarity(tfidf.transform([query_text]), tfidf_matrix).ravel()
    visual_scores = visual_knn_scores(query_image)
    scores = text_weight * text_scores + (1.0 - text_weight) * visual_scores
    return rank_scores(scores, exclude_image_id=exclude_image_id, top_k=top_k)

## 5. Shared Evaluation Queries

Relevance comes from transparent caption keyword rules and optional manual image-id lists. It is not generated from descriptor similarity.

In [ ]:
with QUERY_FILE.open('r', encoding='utf-8') as f:
    evaluation_queries = json.load(f)


def term_in_caption(term, caption):
    return normalize_text(term) in normalize_text(caption)


def captions_match_required(captions, required_terms):
    return all(any(term_in_caption(term, caption) for caption in captions) for term in required_terms)


def relevance_ids(query):
    relevant = set(query.get('manual_relevant_image_ids') or [])
    required_terms = query.get('required_terms') or []
    for _, row in df.iterrows():
        if captions_match_required(row['captions'], required_terms):
            relevant.add(row['image_id'])
    relevant.discard(query.get('reference_image_id'))
    return relevant


def resolve_reference_image(query, relevant):
    reference_id = query.get('reference_image_id')
    if reference_id in image_id_to_pos:
        return reference_id, full_ds[int(image_id_to_pos[reference_id])]['image']
    for candidate in sorted(relevant):
        if candidate in image_id_to_pos:
            warnings.warn(f'Reference image {reference_id} not found in this run; using {candidate} for {query["query_id"]}.')
            return candidate, full_ds[int(image_id_to_pos[candidate])]['image']
    warnings.warn(f'No available reference image for {query["query_id"]}; image/fused mode will be skipped.')
    return None, None


def queries_for_mode(mode):
    return [q for q in evaluation_queries if mode in q.get('query_modes', [])]

print(f'Loaded {len(evaluation_queries)} shared evaluation queries')

## 6. Evaluation Paths

The three functions below intentionally keep text-only, visual-only, and fused evaluation separate.

In [ ]:
def precision_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / k if k else 0.0


def recall_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / len(relevant) if relevant else 0.0


def summarize_metrics(per_query):
    summary = {}
    for k in K_VALUES:
        summary[str(k)] = {
            'avg_precision': float(np.mean([row[f'p@{k}'] for row in per_query])) if per_query else 0.0,
            'avg_recall': float(np.mean([row[f'r@{k}'] for row in per_query])) if per_query else 0.0,
        }
    return summary


def evaluate_text_only(text_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in text_queries:
        relevant = relevance_ids(query)
        retrieved = rank_text_only(query['query_text'], exclude_image_id=query.get('reference_image_id'), top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_visual_only(image_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in image_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = resolve_reference_image(query, relevant)
        if query_image is None:
            continue
        retrieved = rank_visual_knn(query_image, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_fused(dual_queries, alpha=0.5, top_k=max(K_VALUES)):
    per_query = []
    for query in dual_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = resolve_reference_image(query, relevant)
        if query_image is None:
            continue
        retrieved = rank_fused(query['query_text'], query_image, text_weight=alpha, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


text_results = evaluate_text_only(queries_for_mode('text'))
visual_results = evaluate_visual_only(queries_for_mode('image'))
fused_results = evaluate_fused(queries_for_mode('fused'))

classic_results = {
    'fast_dev': FAST_DEV,
    'k_values': K_VALUES,
    'text_only': text_results,
    'visual_only': visual_results,
    'fused': fused_results,
}
print(json.dumps({mode: classic_results[mode]['summary'] for mode in ('text_only', 'visual_only', 'fused')}, indent=2))

## 7. Results / Visualization

Result grids and P@K/R@K plots belong to the results/visualization component.

In [ ]:
def plot_metric_summary(results_by_mode):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for mode, result in results_by_mode.items():
        axes[0].plot(K_VALUES, [result['summary'][str(k)]['avg_precision'] for k in K_VALUES], marker='o', label=mode)
        axes[1].plot(K_VALUES, [result['summary'][str(k)]['avg_recall'] for k in K_VALUES], marker='o', label=mode)
    axes[0].set_title('Average Precision@K')
    axes[1].set_title('Average Recall@K')
    for ax in axes:
        ax.set_xlabel('K')
        ax.grid(True)
        ax.legend()
    fig.tight_layout()
    return fig


fig = plot_metric_summary({'text_only': text_results, 'visual_only': visual_results, 'fused': fused_results})
if not FAST_DEV:
    fig.savefig(FIGURE_DIR / 'classic_precision_recall.png', dpi=150)
plt.show()

result_path = OUTPUT_DIR / ('classic_results_fast_dev.json' if FAST_DEV else 'classic_results_full.json')
result_path.write_text(json.dumps(classic_results, indent=2), encoding='utf-8')
print(f'Saved results to {result_path}')